# Laboratorio 4. Ciclo de vida de los datos

> **Antes de comenzar**
>
> Este laboratorio está diseñado para ejecutarse en **Google Colab**.
>
> Una vez abierto en Google Colab, ejecute la primera celda para clonar el repositorio del curso y acceder a los conjuntos de datos utilizados en este laboratorio.

In [ ]:
!git clone https://github.com/urendacdenisse-hub/ingenieria-de-datos.git

In [ ]:
%cd ingenieria-de-datos

## Configuración inicial

Ejecuta la siguiente celda. Esta configura el entorno de trabajo y carga los recursos necesarios para que los ejemplos funcionen correctamente.

**⚠️ Importante:** No modifiques esta celda.

In [ ]:
# =================================================
# 🚫 NO MODIFICAR ESTA CELDA... O SE CREASHEA
# Configura todo lo necesario para comenzar
# =================================================

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent))

CARPETA_SALIDA = Path("ingenieria-de-datos/datos/salida")
CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)

## En este laboratorio...

Ahora seguiremos el recorrido de esos datos desde el momento en que son capturados hasta que se convierten en información útil para la toma de decisiones. A través de una actividad práctica utilizaremos datos reales recopilados por el grupo para explorar cada una de las etapas del ciclo de vida de los datos.

# Captura de datos

La **captura** consiste en obtener datos desde una fuente.

En este laboratorio, los datos fueron capturados previamente mediante una encuesta.

Puedes volver a consultarla aquí:

🔗 **[Enlace a la encuesta](https://docs.google.com/forms/d/e/1FAIpQLSe_fJb1iVFc5Qa88tDlbaELiD81mNypGjN0X0ks13zOxdk6sg/viewform?usp=sharing&ouid=113475950686840611808)**

# Ingesta de datos

La **ingesta** consiste en incorporar datos desde una fuente a nuestro flujo de trabajo.

Aquí cargaremos en Python las respuestas almacenadas en Google Sheets.

In [ ]:
import pandas as pd

# ==========================================================
# Leyendo csv con datos recopilados atraves de la encuesta
# ==========================================================

URL_ENCUESTA = (
    "https://docs.google.com/spreadsheets/d/e/"
    "2PACX-1vSXE3M3dkSaStM4QkB_gP--ODftAAFg06vUjSSbSi-BdEbz_XuqBVmVBICdBgxU0Nmym10DAQZjW7Mr/"
    "pub?gid=1057677381&single=true&output=csv"
)

datos = pd.read_csv(URL_ENCUESTA)

datos.info()

**Con esto concluye la etapa de ingesta de datos**. Los datos ya se encuentran disponibles en Python y están listos para continuar con la siguiente etapa del ciclo de vida.

# Almacenamiento de datos

El **almacenamiento** permite conservar los datos para utilizarlos posteriormente.

Guardaremos una copia de los datos originales antes de modificarlos.

In [ ]:
# ==========================================================
# Guardando una copia de los datos en formato csv
# ==========================================================

ruta = CARPETA_SALIDA / "retroalimentacion_intro_datos.csv"

datos.to_csv(ruta, index=False)

**A partir de este momento continuaremos trabajando con la variable** `datos`. La copia que acabamos de guardar se conservará como respaldo, por lo que no es necesario volver a cargar el archivo.

# Procesamiento de datos

El **procesamiento** prepara los datos antes de analizarlos.

Realizaremos una inspección, limpieza y selección sencilla.

## Inspección
Revisaremos la estructura, tipos de datos y posibles inconsistencias.

In [ ]:
# ==========================================================
# Conocer la estructura del conjunto de datos:
# - primeros registros
# ==========================================================

datos.head()

In [ ]:
# ==========================================================
# Conocer la estructura del conjunto de datos:
# - Numero de filas (registros) y columns (campos)
# ==========================================================

print(f"Filas: {datos.shape[0]}")
print(f"Columnas: {datos.shape[1]}")

In [ ]:
# ==========================================================
# Identificar los tipos de datos
# ==========================================================

datos.info()

In [ ]:
# ==========================================================
# Detectar posibles inconsistencias:
# - registros duplicados
# ==========================================================

print("Registros duplicados:", datos.duplicated().sum())

In [ ]:
# ==========================================================
# Detectar posibles inconsistencias:
# - número de valores faltantes (NaN/NA)
# ==========================================================

datos.isna().sum()

In [ ]:
# ==========================================================
# Columnas con datos faltantes
# ==========================================================

cuenta_datos_faltantes = datos.isna().sum()
cuenta_datos_faltantes[cuenta_datos_faltantes > 0]

## Limpieza

El tratamiento de valores faltantes depende de su **significado y contexto**.

In [ ]:
cuenta_datos_faltantes[cuenta_datos_faltantes > 0]

No todos los valores faltantes requieren el mismo tratamiento. En este ejemplo conservaremos algunos y reemplazaremos otros según el contexto.

In [ ]:
# ==================================================
# Tratando algunos valores faltantes
# ==================================================

datos_limpios = datos.copy()

datos_limpios["¿Hay algún comentario o sugerencia adicional?"] = (
    datos_limpios["¿Hay algún comentario o sugerencia adicional?"].fillna("Ninguno")
)

datos_limpios.isna().sum()

En este caso **no modificaremos** los valores faltantes de preguntas como:

- *¿Qué fue lo que más te gustó de esta lección?*
- *Si pudieras mejorar algo de esta lección, ¿qué cambiarías?*

No existe una forma de inferir esa información, por lo que es preferible conservar los valores faltantes.

En cambio, para la pregunta:

**¿Hay algún comentario o sugerencia adicional?**

podemos interpretar que un valor faltante significa que el participante **no tenía comentarios adicionales**, por lo que reemplazarlo por un valor como *"Ninguno"* o *"Sin comentarios"* puede facilitar análisis posteriores sin alterar el significado de la respuesta.

## Selección

Seleccionaremos únicamente las variables necesarias para responder nuestra pregunta.

> **Pregunta**: ¿Cuál es la opinión general de los estudiantes sobre la lección?

In [ ]:
# ================================================
# Selección de variables de interés
# ================================================

columnas = [
    "¿Qué tan interesante te pareció esta lección?",
    "¿Qué tan claras fueron las explicaciones presentadas en la lección?",
    "¿Qué tan útiles te parecieron los ejemplos utilizados durante la lección?",
    "¿La actividad final te ayudó a relacionar los conceptos vistos durante la lección?"
]

datos_opinion = datos_limpios[columnas]

datos_opinion.head()

**Observacion**

El conjunto de datos original permanece intacto. Únicamente hemos creado una nueva vista con las variables necesarias para este objetivo.

# Análisis de datos

El **análisis** utiliza los datos para responder preguntas o resolver problemas.

> **Pregunta:** ¿Cuál es la opinión general de los estudiantes sobre la lección?

Nuestro objetivo no es realizar un análisis exhaustivo, sino mostrar cómo los datos pueden utilizarse para obtener información útil.

In [ ]:
# =============================================================================
# Conocer número de respuestas recibidas.
# =============================================================================

print(f"Respuestas recibidas: {len(datos_opinion)}")

In [ ]:
# =============================================================================
# Calculando estadisticas básicas
# =============================================================================

datos_opinion.describe()

In [ ]:
# =============================================================================
# Calculo de media por campo (pregunta)
# =============================================================================

datos_opinion.mean()

# Visualización y comunicación

Utilizaremos una gráfica para interpretar y comunicar los resultados obtenidos.

En esta actividad utilizaremos una visualización sencilla para responder la pregunta planteada anteriormente.

In [ ]:
from recursos.codigo.visualizacion import visualizar_resultados

# ==========================================
# Visualizar los resultados
# ==========================================

figura = visualizar_resultados(datos_opinion)

In [ ]:
# ==========================================
# Crear ruta
# ==========================================

ruta = CARPETA_SALIDA / "resultados_encuesta.png"

# ==========================================
# Guardar los resultados para uso posterior
# ==========================================

figura.savefig(
    fname=ruta,
    dpi=300,
    bbox_inches="tight"
)

print(f"Guardado en: {ruta.resolve()}")

In [ ]:
from recursos.codigo.imagenes import cargar_imagen

# ==========================================
# Comprobar que se guardo correctamente
# ==========================================

cargar_imagen(ruta=ruta)

**Observación:** Una gráfica guardada como imagen también es un dato y puede iniciar su propio ciclo de vida.

# Conservación y eliminación de datos

Al finalizar un proyecto, debemos decidir qué datos **conservar y cuáles eliminar** según su utilidad, privacidad y las políticas aplicables.

## 💡 Por si tienes curiosidad...

Así como es posible guardar archivos desde Python, también es posible eliminarlos de forma automática.

In [ ]:
# =====================================================
# ⚠️ Corre bajo tu propio riesgo.
# =====================================================

ruta = CARPETA_SALIDA / "resultados_encuesta.png"

if ruta.exists():
    ruta.unlink()
    print("🗑️ La imagen fue eliminada.")
else:
    print("⚠️ La imagen ya no existe.")

In [ ]:
print(ruta.exists())

# Resumen del laboratorio

Los datos recorren diferentes etapas desde que son obtenidos hasta que dejan de ser utilizados. A lo largo de este proceso pueden almacenarse, transformarse, analizarse y comunicarse para generar información útil. Además, los resultados obtenidos también pueden convertirse en nuevos datos que inician su propio ciclo de vida.

## Ideas clave:

- El ciclo de vida de los datos está formado por diferentes etapas, cada una con un propósito específico.
- Los datos se transforman continuamente para responder preguntas o resolver problemas.
- Los resultados de un análisis también son datos que pueden almacenarse, compartirse o eliminarse.

# Actividad

En esta actividad aplicarás nuevamente las etapas del ciclo de vida de los datos utilizando un conjunto de datos diferente. Aunque cambiarán las preguntas, los datos y los resultados obtenidos, el proceso seguirá siendo esencialmente el mismo.

👉 Haz clic aquí para abrir la actividad:

**[Actividad 04 - Generación de reporte de resultados](https://colab.research.google.com/github/urendacdenisse-hub/ingenieria-de-datos/blob/main/actividades/unidad-01/actividad-04.ipynb)**